In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import requests
import pandas as pd
from io import BytesIO

In [2]:
SUBCATEGORY_TO_LABEL = {
    "Tops":       0,
    "Bottomwear": 1,
    "Shoes":      2,
    "Dress":      3,
    "Outerwear":  4,
}

# Reverse mapping — useful for interpreting predictions later
LABEL_TO_SUBCATEGORY = {v: k for k, v in SUBCATEGORY_TO_LABEL.items()}

In [3]:
def download_image(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return Image.open(BytesIO(response.content)).convert("RGB")
    except Exception as e:
        print(f"Failed to download {url}: {e}")
        return None


class WardrobeDataset(Dataset):

    def __init__(self, df, local_image_dir=None, image_size=224):
        """
        Args:
            df              : dataframe with columns 'id', 'link', 'subCategory'
            local_image_dir : if images are already saved locally, provide path here
                              to load from disk instead of downloading every time
            image_size      : input size for ResNet (224)
        """
        self.df              = df.reset_index(drop=True)
        self.local_image_dir = local_image_dir

        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],  # ImageNet mean
                std=[0.229, 0.224, 0.225]    # ImageNet std
            )
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row          = self.df.iloc[idx]
        image_id     = row["id"]
        url          = row["link"]
        sub_category = row["subCategory"]
        label        = SUBCATEGORY_TO_LABEL.get(sub_category, -1)

        # Load from disk if available, otherwise download from URL
        if self.local_image_dir:
            local_path = os.path.join(self.local_image_dir, f"{image_id}.jpg")
            img = Image.open(local_path).convert("RGB") if os.path.exists(local_path) else download_image(url)
        else:
            img = download_image(url)

        # Return blank tensor if image failed — won't crash the training loop
        if img is None:
            return torch.zeros(3, 224, 224), label

        # No crop — images are already 80-90% the target item
        img = self.transform(img)
        return img, label

In [8]:
# Shuffle and split combined_df 80/20
combined_df = pd.read_csv("../data/combined_df.csv")

combined_df_shuffled = combined_df.sample(frac=1, random_state=42).reset_index(drop=True).head(250)

split       = int(0.8 * len(combined_df_shuffled))
train_df    = combined_df_shuffled.iloc[:split]
val_df      = combined_df_shuffled.iloc[split:]

print(f"Training samples:   {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

# Create datasets
# If you have images saved locally, pass local_image_dir="data/processed/images/"
# Otherwise leave it out and it will download from URLs on the fly
train_dataset = WardrobeDataset(train_df)
val_dataset   = WardrobeDataset(val_df)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=0)

Training samples:   200
Validation samples: 50


In [9]:
NUM_CLASSES = 5  # Tops, Bottomwear, Shoes, Dress, Outerwear

model = models.resnet50(weights="IMAGENET1K_V1")

# Freeze all layers — only train the final classifier head
for param in model.parameters():
    param.requires_grad = False

# Replace the final layer with one that outputs 5 classes
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

print(f"Using device: {device}")

Using device: cpu


In [10]:
criterion = nn.CrossEntropyLoss()

# Only optimize the final layer since everything else is frozen
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

In [11]:
NUM_EPOCHS = 2

for epoch in range(NUM_EPOCHS):

    # ── Training ──────────────────────────────────────────
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        preds          = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total   += labels.size(0)

    train_acc = train_correct / train_total

    # ── Validation ────────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs        = model(images)
            loss           = criterion(outputs, labels)

            val_loss    += loss.item()
            preds        = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)

    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.4f}")

Epoch 1/2 | Train Loss: 0.9822 | Train Acc: 0.6050 | Val Loss: 0.6601 | Val Acc: 0.8600
Epoch 2/2 | Train Loss: 0.6538 | Train Acc: 0.8950 | Val Loss: 0.5063 | Val Acc: 0.8800


In [12]:
os.makedirs("../models", exist_ok=True)
torch.save(model.state_dict(), "../models/resnet50_stylesync.pt")
print("Model saved to models/resnet50_stylesync.pt")

Model saved to models/resnet50_stylesync.pt
